# Proyecto Final de Machine Learning: Detección de Fraude con Tarjetas de Crédito

En este notebook se desarrolla un proyecto de **detección de transacciones fraudulentas** usando el dataset `creditcard.csv`. La estructura sigue el estilo de las prácticas de la asignatura: explicación clara en celdas de texto, seguida de celdas de código y resultados.

## 1. Introducción

La detección de fraude en transacciones con tarjeta es un problema crítico en entornos financieros. El fraude suele representar una **proporción muy pequeña** de las operaciones, pero con un impacto económico y reputacional enorme. En un escenario real, un sistema de detección debe ser capaz de identificar la mayor cantidad de operaciones fraudulentas posibles **sin generar un número excesivo de falsos positivos**.

**Objetivos del proyecto:**

- Comprender la estructura del dataset y su fuerte desbalance de clases.
- Entrenar y evaluar modelos de clasificación para detectar fraude.
- Comparar un modelo lineal base (Regresión Logística) con un modelo más complejo (Random Forest).
- Usar métricas adecuadas al desbalance (precisión, recall, F1, ROC-AUC).

## 2. Descripción y análisis del dataset

El dataset `creditcard.csv` contiene transacciones reales con tarjetas de crédito. Sus columnas principales son:

- **Time**: segundos transcurridos desde la primera transacción registrada.
- **Amount**: importe de la transacción.
- **V1 – V28**: componentes principales (PCA) ya transformadas para proteger la confidencialidad.
- **Class**: etiqueta objetivo (0 = transacción legítima, 1 = fraude).

Este dataset es un ejemplo típico de **desbalance extremo**, por lo que la evaluación debe centrarse en métricas sensibles a la clase minoritaria.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Carga del dataset
# Asegúrate de que el archivo creditcard.csv esté en el mismo directorio que este notebook

df = pd.read_csv("creditcard.csv")
df.head()

In [ ]:
# Información básica
print(df.shape)
df.info()

In [ ]:
# Distribución de la variable objetivo (desbalance de clases)
class_counts = df["Class"].value_counts()
print(class_counts)

In [ ]:
# Visualización del desbalance
plt.figure(figsize=(6, 4))
sns.countplot(x="Class", data=df)
plt.title("Distribución de clases (0 = legítima, 1 = fraude)")
plt.xlabel("Clase")
plt.ylabel("Número de transacciones")
plt.tight_layout()
plt.show()

**Observación:** la clase fraudulenta es una minoría muy pequeña. En este contexto, usar la **accuracy** puede resultar engañoso: un modelo que prediga siempre la clase 0 tendría una accuracy muy alta, pero sería inútil para detectar fraude.

## 3. Preprocesado de datos

- Separamos variables (X) y etiqueta (y).
- Escalamos variables numéricas para el modelo lineal (especialmente `Amount` y `Time`).
- Dividimos en train/test de forma **estratificada** para mantener la proporción de clases.

In [ ]:
# Separación de variables y etiqueta
X = df.drop(columns=["Class"])
y = df["Class"]

# División train/test estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Escalado de variables numéricas
# En este dataset, todas las variables están ya transformadas excepto Time y Amount
num_features = ["Time", "Amount"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features)
    ],
    remainder="passthrough"
)

## 4. Modelos utilizados

### 4.1 Regresión Logística (modelo base)

La Regresión Logística es un modelo lineal clásico, interpretativo y eficiente. En problemas desbalanceados se puede usar `class_weight='balanced'` para penalizar más los errores sobre la clase minoritaria.

In [ ]:
# Pipeline para Regresión Logística
log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_reg

In [ ]:
# Entrenamiento
log_reg.fit(X_train, y_train)

### 4.2 Random Forest (modelo más complejo)

Random Forest combina múltiples árboles de decisión. Suele capturar relaciones no lineales y ser más potente, pero puede requerir más ajuste. También admite `class_weight='balanced'` para tratar el desbalance.

In [ ]:
# Pipeline para Random Forest
rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf

In [ ]:
# Entrenamiento
rf.fit(X_train, y_train)

## 5. Evaluación de modelos

Se evalúan los modelos con métricas adecuadas para clases desbalanceadas:

- **Precision**: proporción de positivos predichos que son correctos.
- **Recall**: proporción de fraudes reales detectados.
- **F1-score**: equilibrio entre precision y recall.
- **ROC-AUC**: capacidad de discriminación global.
- **Matriz de confusión**: visión completa de errores.

La **accuracy** no es métrica principal en este contexto.

In [ ]:
# Función de evaluación

def evaluar_modelo(modelo, X_test, y_test, nombre="Modelo"):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"=== {nombre} ===")
    print(classification_report(y_test, y_pred, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Matriz de confusión - {nombre}")
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(figsize=(4, 3))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Curva ROC - {nombre}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return precision, recall, f1, roc_auc

In [ ]:
# Evaluación de Regresión Logística
metrics_log = evaluar_modelo(log_reg, X_test, y_test, nombre="Regresión Logística")

In [ ]:
# Evaluación de Random Forest
metrics_rf = evaluar_modelo(rf, X_test, y_test, nombre="Random Forest")

## 6. Comparación de resultados

Compararemos los modelos usando una tabla con las métricas principales.

In [ ]:
# Tabla comparativa
results = pd.DataFrame(
    [metrics_log, metrics_rf],
    columns=["Precision", "Recall", "F1-score", "ROC-AUC"],
    index=["Regresión Logística", "Random Forest"]
)

results

**Análisis comparativo:**

- La **Regresión Logística** es un buen punto de partida, rápida y fácil de interpretar.
- El **Random Forest** suele mejorar la capacidad predictiva en datos complejos, aunque puede ser menos interpretable.
- En problemas de fraude, suele priorizarse un **alto recall** para detectar la mayor cantidad de fraudes posibles.

## 7. Conclusiones

- El modelo con mejor rendimiento debe evaluarse en función de **precision/recall** y **ROC-AUC**, no solo por accuracy.
- En escenarios reales, es preferible capturar la mayor cantidad posible de fraudes (recall alto), aunque eso implique algunos falsos positivos.

**Limitaciones del proyecto:**

- No se han ajustado hiperparámetros de forma exhaustiva.
- Se usa un único split train/test; validaciones más robustas podrían mejorar la evaluación.

**Posibles mejoras futuras:**

- Ajuste fino de hiperparámetros (grid search / random search).
- Uso de técnicas avanzadas de balanceo como **SMOTE**.
- Modelos más complejos (XGBoost, LightGBM) y calibración de probabilidades.